# Data Structures Complete

*Run each cell with **Shift+Enter***

Python Data Structures — Complete Reference (from bool to Graph)
================================================================
Mental model: every data structure answers ONE question:
  "Given a problem shape, which container makes the hot operation O(1)?"

PART 1 : Primitive types      — bool, int, float, complex, str, bytes
PART 2 : Built-in containers  — list, tuple, set, frozenset, dict
PART 3 : collections module   — Counter, defaultdict, deque, OrderedDict,
                                 namedtuple, ChainMap
PART 4 : Stack & Queue        — LIFO / FIFO patterns
PART 5 : Linked List          — singly + doubly + cycle detection
PART 6 : Binary Search Tree   — insert, search, delete, 4 traversals
PART 7 : AVL Tree             — self-balancing BST (LL/RR/LR/RL rotations)
PART 8 : Binary Heap          — min-heap + max-heap; heapq; top-K pattern
PART 9 : Trie                 — insert, search, starts_with, autocomplete
PART 10: Hash Map             — from scratch: chaining collision resolution
PART 11: Graph                — BFS, DFS, Dijkstra, topo sort, cycle detect

Run: python data_structures_complete.py

In [ ]:
from __future__ import annotations

import heapq
import sys
from collections import ChainMap, Counter, OrderedDict, defaultdict, deque, namedtuple
from dataclasses import dataclass
from typing import Any

## PART 1 — PRIMITIVE TYPES
Mental model: primitives are VALUE objects — they are immutable.
  Assigning or passing them around never causes aliasing side effects.

In [ ]:
def demo_primitives() -> None:
    """
    bool  — subclass of int; True == 1, False == 0.
            Use Case: flags, conditionals, short-circuit evaluation.
    """
    assert True + True == 2          # bool IS an int subclass
    assert bool(0) is False
    assert bool("") is False
    assert bool([]) is False
    assert bool(None) is False
    assert bool(42) is True
    assert bool("x") is True
    # Short-circuit: Python stops evaluating as soon as the result is known.
    result = None or "default"       # "default" — None is falsy
    safe   = 0 or 1                  # 1 — picks the first truthy value
    guarded = [] and do_something()  # never calls do_something(); [] is falsy
    assert result == "default" and safe == 1

    """
    int   — arbitrary precision (no overflow in CPython).
            Operations: +, -, *, //, %, **, &, |, ^, ~, <<, >>
            Use Case: counting, indexing, bit manipulation flags.
    """
    assert 10 // 3 == 3              # floor division
    assert 10 % 3  == 1              # modulo (remainder)
    assert 2 ** 10 == 1024           # exponentiation
    assert 0b1010 == 10              # binary literal
    assert 0xFF   == 255             # hex literal
    assert 0o17   == 15              # octal literal
    assert (6 & 3) == 2              # bitwise AND
    assert (6 | 3) == 7              # bitwise OR
    assert (6 ^ 3) == 5              # bitwise XOR
    assert (6 << 1) == 12            # left shift
    assert (6 >> 1) == 3             # right shift
    # Arbitrary precision:
    big = 10 ** 100                  # googol — no overflow, ever
    assert len(str(big)) == 101

    """
    float — IEEE 754 double; ~15 significant digits.
            GOTCHA: 0.1 + 0.2 != 0.3 (binary representation error)
            Use Case: scientific/financial math — but use Decimal for money!
    """
    assert 0.1 + 0.2 != 0.3          # floating-point binary trap
    import decimal
    assert decimal.Decimal("0.1") + decimal.Decimal("0.2") == decimal.Decimal("0.3")
    assert round(0.1 + 0.2, 10) == 0.3        # practical fix: round
    import math
    assert math.isclose(0.1 + 0.2, 0.3)      # idiomatic comparison

    """
    complex — a + bj (j is imaginary unit in Python)
              Use Case: FFT, signal processing, electrical engineering.
    """
    z = 3 + 4j
    assert z.real == 3 and z.imag == 4
    assert abs(z) == 5.0             # magnitude = sqrt(3²+4²)
    assert z.conjugate() == 3 - 4j

    """
    str — immutable sequence of Unicode code points.
          ALL str methods return NEW strings (immutable).
          Key operations (all O(n) unless stated):
    """
    s = "  Hello, World!  "
    assert s.strip()          == "Hello, World!"     # remove whitespace
    assert s.strip().lower()  == "hello, world!"
    assert s.strip().upper()  == "HELLO, WORLD!"
    assert "hello".capitalize() == "Hello"
    assert "Hello World".title()  == "Hello World"
    assert "hello".center(11, "-") == "---hello---"
    assert "hello world".split()  == ["hello", "world"]
    assert ",".join(["a","b","c"]) == "a,b,c"
    assert "hello".replace("l","r") == "herro"
    assert "hello".find("ll")  == 2         # -1 if not found
    assert "hello".index("ll") == 2         # ValueError if not found
    assert "hello".count("l")  == 2
    assert "hello".startswith("he")
    assert "hello".endswith("lo")
    assert "  ".isspace()
    assert "abc123".isalnum()
    assert "123".isdigit()
    assert "abc".isalpha()
    assert "hello {name}".format(name="Ada") == "hello Ada"
    assert f"{'hello':>10}" == "     hello"   # f-string alignment
    assert "hello"[1:3]   == "el"             # slicing: O(k)
    assert "hello"[::-1]  == "olleh"          # reverse string
    # GOTCHA: str + str in a loop is O(n²) — use ''.join(list) instead
    parts = ["a", "b", "c", "d"]
    assert "".join(parts) == "abcd"           # O(n) — correct

    """
    bytes / bytearray — immutable / mutable sequences of 0-255 integers.
                        Use Case: binary data, network protocols, files.
    """
    b = b"hello"
    assert b[0] == 104               # integer, not char
    assert b.hex() == "68656c6c6f"
    assert bytes.fromhex("68656c6c6f") == b"hello"
    ba = bytearray(b"hello")         # mutable version
    ba[0] = 72                       # H
    assert bytes(ba) == b"Hello"

    print("Part 1 (primitives): all assertions passed ✓")

## PART 2 — BUILT-IN CONTAINERS

In [ ]:
def demo_list() -> None:
    """
    list — ordered, mutable, O(1) append/pop at end, O(n) insert/remove at front.
    Mental model: a dynamic array (like Java ArrayList / C++ vector).
                  Index access is O(1); membership test is O(n).
    """
    lst = [3, 1, 4, 1, 5, 9, 2, 6]
    lst.append(7)          # O(1) amortised
    lst.insert(0, 0)       # O(n) — shifts everything right
    lst.remove(1)          # O(n) — removes FIRST occurrence
    assert lst.pop()  == 7 # O(1) pop from end
    assert lst.pop(0) == 0 # O(n) pop from front — use deque for this!
    lst.sort()             # O(n log n) — in-place Timsort
    assert lst == sorted(lst)
    lst.reverse()          # O(n) — in-place
    assert 5 in lst        # O(n) — linear scan
    assert lst.index(5) >= 0   # O(n)
    assert lst.count(1) == 1   # O(n)
    lst2 = lst.copy()          # O(n) shallow copy
    lst.extend([10, 11])       # O(k)
    lst.clear()                # O(n) — drops all references
    assert lst == []

    # Slicing — [start:stop:step], all optional
    nums = list(range(10))     # [0,1,2,...,9]
    assert nums[2:5]    == [2, 3, 4]        # start inclusive, stop exclusive
    assert nums[::2]    == [0,2,4,6,8]      # every other
    assert nums[::-1]   == list(range(9,-1,-1))  # reverse
    assert nums[-3:]    == [7, 8, 9]        # last three

    # List comprehensions (preferred over map/filter)
    squares = [x*x for x in range(5)]
    evens   = [x for x in range(10) if x % 2 == 0]
    flat    = [x for row in [[1,2],[3,4]] for x in row]
    assert squares == [0,1,4,9,16]
    assert evens   == [0,2,4,6,8]
    assert flat    == [1,2,3,4]

    # Sorting with key
    words = ["banana", "apple", "cherry"]
    assert sorted(words)                == ["apple", "banana", "cherry"]
    assert sorted(words, key=len)       == ["apple", "banana", "cherry"]
    assert sorted(words, reverse=True)  == ["cherry", "banana", "apple"]
    points = [(2,3),(1,5),(4,1)]
    assert sorted(points, key=lambda p: p[1]) == [(4,1),(2,3),(1,5)]

    print("Part 2 (list): all assertions passed ✓")


def demo_tuple() -> None:
    """
    tuple — ordered, IMMUTABLE, hashable (if all elements hashable).
    Mental model: a record / database row — fixed structure, known fields.
                  Use as dict keys, function multi-return, namedtuple.
    Key advantage over list: safe to share; usable as dict key / set member.
    """
    t = (1, "hello", 3.14)
    assert t[0] == 1
    assert len(t) == 3
    x, y, z = t                      # tuple unpacking
    assert z == 3.14
    first, *rest = (1, 2, 3, 4)      # starred unpacking
    assert rest == [2, 3, 4]
    # Hashable — can be dict key
    locations = {(40, -74): "New York", (51, 0): "London"}
    assert locations[(40, -74)] == "New York"
    # Single-element tuple needs trailing comma
    singleton = (42,)
    assert isinstance(singleton, tuple)
    # Converting
    as_list = list(t)
    back    = tuple(as_list)
    assert back == t

    print("Part 2 (tuple): all assertions passed ✓")


def demo_set() -> None:
    """
    set — unordered, mutable, O(1) average add/remove/membership.
    Mental model: a hash table with keys only — NO order, NO duplicates.
    Use Case: deduplication, membership tests, set algebra.
    frozenset — immutable set; hashable; can be dict key.
    """
    a, b = {1, 2, 3, 4}, {3, 4, 5, 6}
    assert a & b == {3, 4}            # intersection
    assert a | b == {1,2,3,4,5,6}    # union
    assert a - b == {1, 2}            # difference (in a not b)
    assert a ^ b == {1,2,5,6}        # symmetric difference
    assert {3} <= a                   # issubset
    assert a >= {1, 2}                # issuperset
    assert a.isdisjoint({7, 8})

    s = {3, 1, 4, 1, 5, 9}           # duplicates removed at creation
    assert len(s) == 5
    s.add(2)
    s.discard(99)                     # no error if missing (vs remove())
    s.remove(1)                       # KeyError if missing

    # Deduplication idiom
    data = [1, 2, 2, 3, 3, 3, 4]
    unique = list(dict.fromkeys(data))  # preserves insertion order (3.7+)
    assert unique == [1, 2, 3, 4]

    # frozenset — for nested sets or as dict key
    fs = frozenset([1, 2, 3])
    nested = {fs: "triangle"}         # frozenset is hashable

    print("Part 2 (set/frozenset): all assertions passed ✓")


def demo_dict() -> None:
    """
    dict — key→value mapping, O(1) average get/set/delete.
    Mental model: a hash table — the most versatile data structure in Python.
    In Python 3.7+ dicts are ORDERED by insertion order.
    Use Case: lookup tables, caches, frequency maps, JSON-like structures.
    """
    d = {"a": 1, "b": 2, "c": 3}
    assert d["a"] == 1
    assert d.get("z", 0) == 0         # safe get with default
    assert "b" in d                   # O(1) key membership
    d["d"] = 4                        # insert/update O(1)
    del d["a"]                        # delete O(1)
    assert d.pop("b", None) == 2      # pop with default
    assert d.setdefault("e", 5) == 5  # get or set if missing
    d.update({"f": 6, "g": 7})
    assert list(d.keys())   == ["c","d","e","f","g"]
    assert list(d.values()) == [3,4,5,6,7]
    pairs = list(d.items())
    assert ("c", 3) in pairs

    # Dict comprehension
    sq = {n: n*n for n in range(5)}
    assert sq[3] == 9

    # Merging dicts (Python 3.9+)
    x = {"a": 1}; y = {"b": 2}
    merged = x | y                    # new dict — does not modify x or y
    assert merged == {"a": 1, "b": 2}

    # Accessing nested dicts safely
    config = {"db": {"host": "localhost", "port": 5432}}
    host = config.get("db", {}).get("host", "unknown")
    assert host == "localhost"

    print("Part 2 (dict): all assertions passed ✓")

## PART 3 — COLLECTIONS MODULE

In [ ]:
def demo_collections() -> None:
    """
    Counter  — frequency map, most_common(), arithmetic
    defaultdict — missing key creates default value, no KeyError
    deque    — O(1) append/pop at BOTH ends
    OrderedDict — dict with move_to_end; remember insertion order explicitly
    namedtuple — lightweight struct with field names
    ChainMap — union of dicts without copying, parent scopes
    """
    # Counter
    c = Counter("abracadabra")
    assert c["a"] == 5
    assert c.most_common(2) == [("a",5),("b",2)]
    c2 = Counter("abc")
    assert (c + c2)["a"] == 6         # addition
    assert (c - c2)["a"] == 4         # subtraction (drops zeros/negatives)

    # defaultdict
    graph = defaultdict(list)
    for u, v in [("A","B"),("A","C"),("B","D")]:
        graph[u].append(v)
    assert graph["A"] == ["B","C"]
    assert graph["X"] == []           # no KeyError — creates empty list

    word_count = defaultdict(int)
    for word in "the cat sat on the mat".split():
        word_count[word] += 1
    assert word_count["the"] == 2

    # deque — O(1) both ends; list.insert(0) is O(n)!
    dq = deque([1,2,3], maxlen=5)     # bounded; oldest dropped on overflow
    dq.appendleft(0)                  # O(1)
    dq.append(4)                      # O(1)
    assert list(dq) == [0,1,2,3,4]
    assert dq.popleft() == 0          # O(1)
    dq.rotate(2)                      # rotate right by 2
    assert list(dq) == [3,4,1,2]

    # OrderedDict
    od = OrderedDict()
    od["first"] = 1; od["second"] = 2; od["third"] = 3
    od.move_to_end("first")           # move to end
    assert list(od) == ["second","third","first"]
    od.move_to_end("first", last=False)  # move to front
    assert list(od.keys())[0] == "first"

    # namedtuple — immutable, attribute access, memory-efficient
    Point = namedtuple("Point", ["x","y"])
    p = Point(3, 4)
    assert p.x == 3 and p.y == 4
    assert p._asdict() == {"x":3,"y":4}
    p2 = p._replace(x=10)            # creates new instance
    assert p2.x == 10 and p.x == 3   # original unchanged

    # ChainMap — layer dicts without copying (parent scope lookup)
    defaults = {"color":"red","size":"medium","font":"Arial"}
    user_prefs = {"color":"blue"}
    session = {"size":"large"}
    combined = ChainMap(session, user_prefs, defaults)
    assert combined["color"] == "blue"    # user_prefs wins
    assert combined["size"]  == "large"  # session wins
    assert combined["font"]  == "Arial"  # falls through to defaults
    combined["new_key"] = "value"        # writes to FIRST map only
    assert "new_key" in session

    print("Part 3 (collections): all assertions passed ✓")

## PART 4 — STACK & QUEUE
Mental model:
  Stack = a pile of plates — Last In, First Out (LIFO)
  Queue = a line of people — First In, First Out (FIFO)

In [ ]:
class Stack:
    """
    LIFO — Last In First Out.
    Operations: push O(1), pop O(1), peek O(1), is_empty O(1).
    Use Cases: undo/redo, call stack, DFS, balanced brackets, expression eval.
    """
    def __init__(self) -> None:
        self._data: list[Any] = []

    def push(self, item: Any) -> None:  self._data.append(item)
    def pop(self)             -> Any:   return self._data.pop()
    def peek(self)            -> Any:   return self._data[-1]
    def is_empty(self)        -> bool:  return len(self._data) == 0
    def __len__(self)         -> int:   return len(self._data)

    def __repr__(self) -> str:
        return f"Stack({self._data})"


def is_balanced(s: str) -> bool:
    """Validate bracket matching using a stack. O(n) time, O(n) space."""
    pairs = {")":"(", "]":"[", "}":"{"}
    stack: list[str] = []
    for ch in s:
        if ch in "([{":
            stack.append(ch)
        elif ch in ")]}":
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return len(stack) == 0


class Queue:
    """
    FIFO — First In First Out.
    Backed by deque for O(1) enqueue AND dequeue.
    Use Cases: BFS, task scheduling, producer-consumer, rate limiting.
    """
    def __init__(self) -> None:
        self._data: deque[Any] = deque()

    def enqueue(self, item: Any) -> None: self._data.append(item)
    def dequeue(self)            -> Any:  return self._data.popleft()
    def front(self)              -> Any:  return self._data[0]
    def is_empty(self)           -> bool: return len(self._data) == 0
    def __len__(self)            -> int:  return len(self._data)


class CircularBuffer:
    """
    Fixed-size ring buffer; oldest element overwritten when full.
    Use Case: sliding window, sensor data, log streaming.
    deque(maxlen=N) does this natively — use that in production.
    """
    def __init__(self, capacity: int) -> None:
        self._buf: list[Any] = [None] * capacity
        self._cap = capacity
        self._head = self._tail = self._size = 0

    def write(self, item: Any) -> None:
        self._buf[self._tail] = item
        self._tail = (self._tail + 1) % self._cap
        if self._size < self._cap:
            self._size += 1
        else:
            self._head = (self._head + 1) % self._cap  # overwrite oldest

    def read(self) -> Any:
        if self._size == 0:
            raise IndexError("buffer empty")
        val = self._buf[self._head]
        self._head = (self._head + 1) % self._cap
        self._size -= 1
        return val

    @property
    def full(self) -> bool: return self._size == self._cap


def demo_stack_queue() -> None:
    s = Stack()
    for i in range(5): s.push(i)
    assert s.peek() == 4 and len(s) == 5
    assert s.pop() == 4 and len(s) == 4

    assert is_balanced("({[]})")
    assert not is_balanced("({[}])")
    assert is_balanced("")

    q = Queue()
    for i in range(3): q.enqueue(i)
    assert q.dequeue() == 0
    assert q.front() == 1

    cb = CircularBuffer(3)
    for v in [1,2,3,4]: cb.write(v)
    assert cb.full
    assert cb.read() == 2   # 1 was overwritten

    print("Part 4 (Stack/Queue): all assertions passed ✓")

## PART 5 — LINKED LISTS
Mental model: a chain of nodes, each knowing its next neighbor.
  Advantage over list: O(1) insert/delete at known position.
  Disadvantage: O(n) random access (no index); poor cache locality.

In [ ]:
class _SNode:
    __slots__ = ("val", "next")
    def __init__(self, val: Any, nxt: "_SNode | None" = None):
        self.val, self.next = val, nxt


class SinglyLinkedList:
    """
    Singly linked: each node knows only next.
    Supported: append O(1)*, prepend O(1), search O(n), delete O(n),
               reverse O(n), has_cycle O(n), length O(1).
    *append is O(1) because we maintain a tail pointer.
    """
    def __init__(self) -> None:
        self._head: _SNode | None = None
        self._tail: _SNode | None = None
        self._len = 0

    def append(self, val: Any) -> None:
        node = _SNode(val)
        if self._tail:
            self._tail.next = node
        else:
            self._head = node
        self._tail = node
        self._len += 1

    def prepend(self, val: Any) -> None:
        node = _SNode(val, self._head)
        self._head = node
        if self._tail is None:
            self._tail = node
        self._len += 1

    def delete_val(self, val: Any) -> bool:
        prev, curr = None, self._head
        while curr:
            if curr.val == val:
                if prev:
                    prev.next = curr.next
                else:
                    self._head = curr.next
                if curr is self._tail:
                    self._tail = prev
                self._len -= 1
                return True
            prev, curr = curr, curr.next
        return False

    def to_list(self) -> list[Any]:
        out, node = [], self._head
        while node:
            out.append(node.val); node = node.next
        return out

    def reverse(self) -> None:
        """Reverse in-place: O(n) time, O(1) space."""
        prev, curr = None, self._head
        self._tail = self._head
        while curr:
            nxt = curr.next
            curr.next = prev
            prev, curr = curr, nxt
        self._head = prev

    def has_cycle(self) -> bool:
        """Floyd's two-pointer (tortoise and hare). O(n), O(1)."""
        slow = fast = self._head
        while fast and fast.next:
            slow = slow.next           # type: ignore[assignment]
            fast = fast.next.next
            if slow is fast:
                return True
        return False

    def __len__(self) -> int: return self._len


class _DNode:
    __slots__ = ("val", "prev", "next")
    def __init__(self, val: Any):
        self.val, self.prev, self.next = val, None, None


class DoublyLinkedList:
    """
    Doubly linked: each node knows prev AND next.
    Advantage: O(1) delete with only a reference to the node (no search!).
    Use Case: LRU cache (linked list + hash map), undo/redo stack.
    """
    def __init__(self) -> None:
        # Sentinel head/tail nodes eliminate edge-case null checks.
        self._head, self._tail = _DNode(None), _DNode(None)
        self._head.next = self._tail
        self._tail.prev = self._head
        self._size = 0

    def _link(self, prev: _DNode, node: _DNode, nxt: _DNode) -> None:
        prev.next = node; node.prev = prev
        node.next = nxt;  nxt.prev  = node

    def append(self, val: Any) -> _DNode:
        node = _DNode(val)
        self._link(self._tail.prev, node, self._tail)  # type: ignore[arg-type]
        self._size += 1
        return node

    def prepend(self, val: Any) -> _DNode:
        node = _DNode(val)
        self._link(self._head, node, self._head.next)  # type: ignore[arg-type]
        self._size += 1
        return node

    def remove(self, node: _DNode) -> None:
        """O(1) — remove a node given its reference."""
        node.prev.next = node.next                     # type: ignore[union-attr]
        node.next.prev = node.prev                     # type: ignore[union-attr]
        self._size -= 1

    def to_list(self) -> list[Any]:
        out, cur = [], self._head.next
        while cur is not self._tail:
            out.append(cur.val); cur = cur.next        # type: ignore[assignment]
        return out

    def __len__(self) -> int: return self._size


def demo_linked_lists() -> None:
    ll = SinglyLinkedList()
    for v in [1,2,3,4,5]: ll.append(v)
    assert ll.to_list() == [1,2,3,4,5]
    ll.prepend(0)
    assert ll.to_list()[0] == 0
    ll.delete_val(3)
    assert 3 not in ll.to_list()
    ll.reverse()
    assert ll.to_list()[0] == 5
    assert not ll.has_cycle()

    dll = DoublyLinkedList()
    n = dll.append(10)
    dll.append(20)
    dll.prepend(5)
    assert dll.to_list() == [5,10,20]
    dll.remove(n)
    assert dll.to_list() == [5,20]

    print("Part 5 (LinkedList): all assertions passed ✓")

## PART 6 — BINARY SEARCH TREE (BST)
Mental model: at each node, everything LEFT is smaller, RIGHT is larger.
  Average case: O(log n) search/insert/delete.
  Worst case (sorted input): O(n) — degenerates to linked list! → Use AVL.

In [ ]:
class BSTNode:
    __slots__ = ("key","val","left","right")
    def __init__(self, key: Any, val: Any = None):
        self.key, self.val = key, val
        self.left: BSTNode | None = None
        self.right: BSTNode | None = None


class BST:
    """
    Operations:
      insert    O(h) — h = height; O(log n) avg, O(n) worst
      search    O(h)
      delete    O(h)
      min/max   O(h)
      successor O(h)
      traversals: inorder (sorted), preorder, postorder, level-order
    """
    def __init__(self) -> None:
        self._root: BSTNode | None = None
        self._size = 0

    # ── insert ─────────────────────────────────────────────────────────────
    def insert(self, key: Any, val: Any = None) -> None:
        self._root = self._ins(self._root, key, val)

    def _ins(self, node: BSTNode | None, key: Any, val: Any) -> BSTNode:
        if node is None:
            self._size += 1
            return BSTNode(key, val)
        if   key < node.key: node.left  = self._ins(node.left,  key, val)
        elif key > node.key: node.right = self._ins(node.right, key, val)
        else:                node.val   = val        # update existing
        return node

    # ── search ─────────────────────────────────────────────────────────────
    def search(self, key: Any) -> Any | None:
        node = self._root
        while node:
            if   key < node.key: node = node.left
            elif key > node.key: node = node.right
            else:                return node.val
        return None

    def __contains__(self, key: Any) -> bool:
        return self.search(key) is not None

    # ── min / max ───────────────────────────────────────────────────────────
    def min_key(self) -> Any:
        n = self._root
        while n and n.left: n = n.left
        return n.key if n else None

    def max_key(self) -> Any:
        n = self._root
        while n and n.right: n = n.right
        return n.key if n else None

    # ── delete ─────────────────────────────────────────────────────────────
    def delete(self, key: Any) -> None:
        self._root, deleted = self._del(self._root, key)
        if deleted: self._size -= 1

    def _del(self, node: BSTNode | None, key: Any) -> tuple[BSTNode | None, bool]:
        if node is None: return None, False
        deleted = False
        if key < node.key:
            node.left,  deleted = self._del(node.left,  key)
        elif key > node.key:
            node.right, deleted = self._del(node.right, key)
        else:
            deleted = True
            if node.left is None:  return node.right, deleted
            if node.right is None: return node.left,  deleted
            # Two children: replace with in-order successor (min of right subtree)
            succ = node.right
            while succ.left: succ = succ.left
            node.key, node.val = succ.key, succ.val
            node.right, _ = self._del(node.right, succ.key)
        return node, deleted

    # ── traversals ─────────────────────────────────────────────────────────
    def inorder(self) -> list[Any]:
        """Left → Root → Right  →  yields SORTED keys."""
        out: list[Any] = []
        def walk(n: BSTNode | None) -> None:
            if n:
                walk(n.left); out.append(n.key); walk(n.right)
        walk(self._root); return out

    def preorder(self) -> list[Any]:
        """Root → Left → Right  →  useful for copying / serializing."""
        out: list[Any] = []
        def walk(n: BSTNode | None) -> None:
            if n:
                out.append(n.key); walk(n.left); walk(n.right)
        walk(self._root); return out

    def postorder(self) -> list[Any]:
        """Left → Right → Root  →  useful for deletion."""
        out: list[Any] = []
        def walk(n: BSTNode | None) -> None:
            if n:
                walk(n.left); walk(n.right); out.append(n.key)
        walk(self._root); return out

    def level_order(self) -> list[list[Any]]:
        """BFS level by level — returns list of levels."""
        if not self._root: return []
        levels, q = [], deque([self._root])
        while q:
            level = []
            for _ in range(len(q)):
                n = q.popleft()
                level.append(n.key)
                if n.left:  q.append(n.left)
                if n.right: q.append(n.right)
            levels.append(level)
        return levels

    def height(self) -> int:
        def _h(n: BSTNode | None) -> int:
            return 0 if n is None else 1 + max(_h(n.left), _h(n.right))
        return _h(self._root)

    def __len__(self) -> int: return self._size


def demo_bst() -> None:
    bst = BST()
    keys = [5, 3, 7, 1, 4, 6, 8, 2]
    for k in keys: bst.insert(k, f"v{k}")
    assert bst.inorder()   == [1,2,3,4,5,6,7,8]  # sorted!
    assert bst.preorder()  == [5,3,1,2,4,7,6,8]
    assert bst.postorder() == [2,1,4,3,6,8,7,5]
    assert bst.level_order() == [[5],[3,7],[1,4,6,8],[2]]
    assert bst.min_key() == 1 and bst.max_key() == 8
    assert 5 in bst and 9 not in bst
    bst.delete(3)
    assert 3 not in bst
    assert bst.inorder() == [1,2,4,5,6,7,8]

    # GOTCHA: sorted input degenerates BST to O(n) height
    degenerate = BST()
    for i in range(1, 11): degenerate.insert(i)
    assert degenerate.height() == 10  # chain, not tree!

    print("Part 6 (BST): all assertions passed ✓")

## PART 7 — AVL TREE (Self-Balancing BST)
Mental model: BST that rotates after every insert/delete to keep
  |height(left) - height(right)| ≤ 1 at every node.
  Guarantees O(log n) for ALL operations, even with sorted input.

In [ ]:
class _AVLNode:
    __slots__ = ("key","left","right","height")
    def __init__(self, key: Any):
        self.key = key
        self.left: _AVLNode | None  = None
        self.right: _AVLNode | None = None
        self.height = 1


class AVLTree:
    """
    Four rotation cases:
      LL: right-heavy left child  → right rotate
      RR: left-heavy  right child → left rotate
      LR: right-heavy left child  → left rotate child, then right rotate
      RL: left-heavy  right child → right rotate child, then left rotate
    All operations: O(log n) guaranteed.
    """
    def __init__(self) -> None:
        self._root: _AVLNode | None = None
        self._size = 0

    @staticmethod
    def _h(n: _AVLNode | None) -> int:
        return n.height if n else 0

    @staticmethod
    def _bf(n: _AVLNode | None) -> int:
        return AVLTree._h(n.left) - AVLTree._h(n.right) if n else 0

    @staticmethod
    def _upd(n: _AVLNode) -> None:
        n.height = 1 + max(AVLTree._h(n.left), AVLTree._h(n.right))

    @staticmethod
    def _rr(y: _AVLNode) -> _AVLNode:
        """Right rotation."""
        x = y.left; assert x
        y.left = x.right; x.right = y
        AVLTree._upd(y); AVLTree._upd(x); return x

    @staticmethod
    def _lr(x: _AVLNode) -> _AVLNode:
        """Left rotation."""
        y = x.right; assert y
        x.right = y.left; y.left = x
        AVLTree._upd(x); AVLTree._upd(y); return y

    def _balance(self, n: _AVLNode) -> _AVLNode:
        self._upd(n)
        bf = self._bf(n)
        if bf > 1:                               # Left heavy
            if self._bf(n.left) < 0:            # LR case
                n.left = self._lr(n.left)        # type: ignore[arg-type]
            return self._rr(n)
        if bf < -1:                              # Right heavy
            if self._bf(n.right) > 0:           # RL case
                n.right = self._rr(n.right)     # type: ignore[arg-type]
            return self._lr(n)
        return n

    def insert(self, key: Any) -> None:
        self._root = self._ins(self._root, key)
        self._size += 1

    def _ins(self, n: _AVLNode | None, key: Any) -> _AVLNode:
        if n is None: return _AVLNode(key)
        if   key < n.key: n.left  = self._ins(n.left,  key)
        elif key > n.key: n.right = self._ins(n.right, key)
        else:             self._size -= 1   # duplicate — undo size increment
        return self._balance(n)

    def inorder(self) -> list[Any]:
        out: list[Any] = []
        def walk(n: _AVLNode | None) -> None:
            if n:
                walk(n.left); out.append(n.key); walk(n.right)
        walk(self._root); return out

    def height(self) -> int:
        return self._h(self._root)

    def __len__(self) -> int: return self._size


def demo_avl() -> None:
    avl = AVLTree()
    # Insert sorted — plain BST would be O(n) height; AVL stays O(log n)
    for i in range(1, 17):
        avl.insert(i)
    import math
    assert avl.height() <= math.ceil(math.log2(16)) + 1
    assert avl.inorder() == list(range(1, 17))
    assert len(avl) == 16
    print("Part 7 (AVL): all assertions passed ✓")

## PART 8 — BINARY HEAP
Mental model: a COMPLETE binary tree stored as an array, satisfying the
  HEAP PROPERTY: every node ≤ (min-heap) or ≥ (max-heap) its children.
  The root is always the minimum (or maximum) element.
  Operations: push O(log n), pop O(log n), peek O(1).
  Use Case: priority queues, Dijkstra, Prim's, top-K problems.

In [ ]:
class MinHeap:
    """
    Min-heap implemented as a 0-indexed array.
    Parent of i: (i-1)//2
    Left child:  2*i + 1
    Right child: 2*i + 2
    """
    def __init__(self) -> None:
        self._data: list[Any] = []

    def push(self, item: Any) -> None:
        self._data.append(item)
        self._sift_up(len(self._data) - 1)

    def pop(self) -> Any:
        if len(self._data) == 1:
            return self._data.pop()
        root = self._data[0]
        self._data[0] = self._data.pop()  # move last to root
        self._sift_down(0)
        return root

    def peek(self) -> Any:
        return self._data[0]

    def _sift_up(self, i: int) -> None:
        while i > 0:
            parent = (i - 1) // 2
            if self._data[i] < self._data[parent]:
                self._data[i], self._data[parent] = self._data[parent], self._data[i]
                i = parent
            else:
                break

    def _sift_down(self, i: int) -> None:
        n = len(self._data)
        while True:
            smallest, l, r = i, 2*i+1, 2*i+2
            if l < n and self._data[l] < self._data[smallest]: smallest = l
            if r < n and self._data[r] < self._data[smallest]: smallest = r
            if smallest == i: break
            self._data[i], self._data[smallest] = self._data[smallest], self._data[i]
            i = smallest

    def __len__(self) -> int: return len(self._data)


def top_k_smallest(nums: list[int], k: int) -> list[int]:
    """Return k smallest using a max-heap of size k. O(n log k)."""
    # Trick: negate to use min-heap as max-heap
    heap: list[int] = []
    for n in nums:
        heapq.heappush(heap, -n)
        if len(heap) > k:
            heapq.heappop(heap)
    return sorted(-x for x in heap)


def demo_heap() -> None:
    h = MinHeap()
    for v in [5, 3, 8, 1, 4]:
        h.push(v)
    assert h.peek() == 1
    result = [h.pop() for _ in range(len(h))]
    assert result == [1,3,4,5,8]   # sorted ascending

    # Python's heapq — same behaviour, built-in
    import heapq
    data = [5,3,8,1,4,2]
    heap = data.copy()
    heapq.heapify(heap)             # O(n) in-place
    assert heapq.heappop(heap) == 1

    # Top-K
    nums = [7, 10, 4, 3, 20, 15]
    assert top_k_smallest(nums, 3) == [3, 4, 7]

    # Max-heap via negation (heapq is min-heap only)
    max_heap: list[int] = []
    for v in [5, 3, 8, 1]: heapq.heappush(max_heap, -v)
    assert -heapq.heappop(max_heap) == 8   # maximum

    print("Part 8 (Heap): all assertions passed ✓")

## PART 9 — TRIE (Prefix Tree)
Mental model: a tree where each PATH from root to a node spells a word prefix.
  insert O(m), search O(m), starts_with O(m) — m = key length.
  Use Case: autocomplete, spell checkers, IP routing, word games.

In [ ]:
class TrieNode:
    __slots__ = ("children","is_end","count")
    def __init__(self) -> None:
        self.children: dict[str, TrieNode] = {}
        self.is_end = False
        self.count  = 0    # number of words passing through this node


class Trie:
    def __init__(self) -> None:
        self._root = TrieNode()

    def insert(self, word: str) -> None:
        node = self._root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
            node.count += 1
        node.is_end = True

    def search(self, word: str) -> bool:
        """Returns True if EXACT word exists."""
        node = self._root
        for ch in word:
            if ch not in node.children: return False
            node = node.children[ch]
        return node.is_end

    def starts_with(self, prefix: str) -> bool:
        """Returns True if any word starts with prefix."""
        node = self._root
        for ch in prefix:
            if ch not in node.children: return False
            node = node.children[ch]
        return True

    def count_prefix(self, prefix: str) -> int:
        """How many inserted words start with this prefix."""
        node = self._root
        for ch in prefix:
            if ch not in node.children: return 0
            node = node.children[ch]
        return node.count

    def autocomplete(self, prefix: str) -> list[str]:
        """Return all words starting with prefix."""
        node = self._root
        for ch in prefix:
            if ch not in node.children: return []
            node = node.children[ch]
        results: list[str] = []
        self._dfs(node, list(prefix), results)
        return sorted(results)

    def _dfs(self, node: TrieNode, path: list[str], results: list[str]) -> None:
        if node.is_end: results.append("".join(path))
        for ch, child in node.children.items():
            path.append(ch)
            self._dfs(child, path, results)
            path.pop()

    def delete(self, word: str) -> bool:
        def _del(node: TrieNode, word: str, depth: int) -> bool:
            if depth == len(word):
                if not node.is_end: return False
                node.is_end = False
                return len(node.children) == 0  # can delete node?
            ch = word[depth]
            if ch not in node.children: return False
            child = node.children[ch]
            should_delete = _del(child, word, depth + 1)
            if should_delete:
                del node.children[ch]
                return not node.is_end and len(node.children) == 0
            return False
        return _del(self._root, word, 0)


def demo_trie() -> None:
    t = Trie()
    words = ["apple","app","application","apply","banana","band"]
    for w in words: t.insert(w)
    assert t.search("apple")
    assert not t.search("appl")
    assert t.starts_with("appl")
    assert t.count_prefix("app") == 4
    ac = t.autocomplete("app")
    assert ac == ["app","apple","application","apply"]
    t.delete("apple")
    assert not t.search("apple")
    assert t.search("app")    # prefix still exists

    print("Part 9 (Trie): all assertions passed ✓")

## PART 10 — HASH MAP FROM SCRATCH
Mental model: array + hash function → O(1) average key operations.
  Collision resolution: separate chaining (each slot = linked list / Python list).
  Load factor > 0.7 → resize (rehash) to keep O(1) amortised.

In [ ]:
class HashMap:
    """
    Separate chaining hash map.
    Average: get/put/delete O(1).   Worst (all same hash): O(n).
    Amortised: O(1) due to automatic resizing.
    """
    _INITIAL_CAPACITY = 8
    _LOAD_THRESHOLD   = 0.7

    def __init__(self) -> None:
        self._capacity = self._INITIAL_CAPACITY
        self._buckets: list[list] = [[] for _ in range(self._capacity)]
        self._size = 0

    def _bucket_index(self, key: Any) -> int:
        return hash(key) % self._capacity

    def put(self, key: Any, val: Any) -> None:
        idx = self._bucket_index(key)
        for pair in self._buckets[idx]:
            if pair[0] == key:
                pair[1] = val; return
        self._buckets[idx].append([key, val])
        self._size += 1
        if self._size / self._capacity > self._LOAD_THRESHOLD:
            self._resize()

    def get(self, key: Any, default: Any = None) -> Any:
        for pair in self._buckets[self._bucket_index(key)]:
            if pair[0] == key: return pair[1]
        return default

    def delete(self, key: Any) -> bool:
        idx = self._bucket_index(key)
        bucket = self._buckets[idx]
        for i, pair in enumerate(bucket):
            if pair[0] == key:
                bucket.pop(i); self._size -= 1; return True
        return False

    def _resize(self) -> None:
        old = self._buckets
        self._capacity *= 2
        self._buckets = [[] for _ in range(self._capacity)]
        self._size = 0
        for bucket in old:
            for key, val in bucket:
                self.put(key, val)

    def __contains__(self, key: Any) -> bool:
        return self.get(key) is not None

    def __len__(self) -> int: return self._size

    def keys(self) -> list[Any]:
        return [pair[0] for b in self._buckets for pair in b]


def demo_hashmap() -> None:
    hm = HashMap()
    for i in range(20):
        hm.put(f"key{i}", i * 10)
    assert hm.get("key5") == 50
    assert "key10" in hm
    hm.delete("key5")
    assert hm.get("key5") is None
    assert len(hm) == 19
    print("Part 10 (HashMap): all assertions passed ✓")

## PART 11 — GRAPH
Mental model: nodes (vertices) + connections (edges).
  Directed vs Undirected, Weighted vs Unweighted, Cyclic vs Acyclic.
  Representation: adjacency list (sparse) or matrix (dense).
  BFS → shortest path (unweighted) / level-order exploration.
  DFS → connectivity / topological sort / cycle detection.
  Dijkstra → shortest path (non-negative weights).

In [ ]:
class Graph:
    """
    Adjacency-list representation.
    directed=True  → edges one-way
    directed=False → edges both ways (undirected)
    """
    def __init__(self, directed: bool = False) -> None:
        self._adj: dict[Any, list[tuple[Any, float]]] = defaultdict(list)
        self._directed = directed

    def add_edge(self, u: Any, v: Any, weight: float = 1.0) -> None:
        self._adj[u].append((v, weight))
        if not self._directed:
            self._adj[v].append((u, weight))
        # Ensure v appears as a node even if it has no outgoing edges
        if v not in self._adj:
            self._adj[v] = []

    def add_node(self, u: Any) -> None:
        if u not in self._adj: self._adj[u] = []

    def neighbors(self, u: Any) -> list[tuple[Any, float]]:
        return self._adj.get(u, [])

    def nodes(self) -> list[Any]:
        return list(self._adj)

    # ── BFS ─────────────────────────────────────────────────────────────────
    def bfs(self, start: Any) -> list[Any]:
        """Breadth-first traversal. Returns nodes in visited order."""
        seen, order, q = {start}, [], deque([start])
        while q:
            u = q.popleft(); order.append(u)
            for v, _ in self._adj[u]:
                if v not in seen:
                    seen.add(v); q.append(v)
        return order

    def shortest_path_bfs(self, src: Any, dst: Any) -> list[Any]:
        """Shortest path (fewest edges) using BFS. O(V+E)."""
        if src == dst: return [src]
        parent: dict[Any, Any] = {src: None}
        q = deque([src])
        while q:
            u = q.popleft()
            for v, _ in self._adj[u]:
                if v not in parent:
                    parent[v] = u; q.append(v)
                    if v == dst:
                        path = []
                        while v is not None:
                            path.append(v); v = parent[v]
                        return path[::-1]
        return []   # no path

    # ── DFS ─────────────────────────────────────────────────────────────────
    def dfs(self, start: Any) -> list[Any]:
        """Iterative DFS traversal."""
        seen, order, stack = set(), [], [start]
        while stack:
            u = stack.pop()
            if u in seen: continue
            seen.add(u); order.append(u)
            for v, _ in reversed(self._adj[u]):
                if v not in seen: stack.append(v)
        return order

    def has_cycle_directed(self) -> bool:
        """Detect cycle in directed graph via DFS with color marking."""
        WHITE, GRAY, BLACK = 0, 1, 2
        color = {n: WHITE for n in self._adj}
        def dfs_color(u: Any) -> bool:
            color[u] = GRAY
            for v, _ in self._adj[u]:
                if color[v] == GRAY: return True    # back edge = cycle
                if color[v] == WHITE and dfs_color(v): return True
            color[u] = BLACK; return False
        return any(dfs_color(n) for n in self._adj if color[n] == WHITE)

    # ── TOPOLOGICAL SORT (Kahn's BFS) ───────────────────────────────────────
    def topological_sort(self) -> list[Any]:
        """Kahn's algorithm (BFS). O(V+E). Returns [] if cycle exists."""
        in_degree = {n: 0 for n in self._adj}
        for u in self._adj:
            for v, _ in self._adj[u]:
                in_degree[v] += 1
        q = deque(n for n, d in in_degree.items() if d == 0)
        order: list[Any] = []
        while q:
            u = q.popleft(); order.append(u)
            for v, _ in self._adj[u]:
                in_degree[v] -= 1
                if in_degree[v] == 0: q.append(v)
        return order if len(order) == len(self._adj) else []

    # ── DIJKSTRA ────────────────────────────────────────────────────────────
    def dijkstra(self, src: Any) -> tuple[dict[Any,float], dict[Any,Any]]:
        """
        Shortest paths from src (non-negative weights only).
        Returns (dist, prev) where prev lets you reconstruct paths.
        O((V + E) log V) with a binary heap.
        """
        INF  = float("inf")
        dist = {n: INF for n in self._adj}; dist[src] = 0
        prev: dict[Any, Any] = {}
        heap: list[tuple[float,Any]] = [(0, src)]
        while heap:
            d, u = heapq.heappop(heap)
            if d > dist[u]: continue    # stale entry
            for v, w in self._adj[u]:
                if dist[u] + w < dist[v]:
                    dist[v] = dist[u] + w
                    prev[v] = u
                    heapq.heappush(heap, (dist[v], v))
        return dist, prev

    def path_to(self, prev: dict, src: Any, dst: Any) -> list[Any]:
        path, v = [], dst
        while v != src:
            path.append(v); v = prev.get(v)
            if v is None: return []
        path.append(src); return path[::-1]

    # ── CONNECTED COMPONENTS ────────────────────────────────────────────────
    def connected_components(self) -> list[list[Any]]:
        """For undirected graphs. O(V+E)."""
        seen: set[Any] = set()
        components: list[list[Any]] = []
        for node in self._adj:
            if node not in seen:
                comp = self.bfs(node)
                components.append(comp)
                seen.update(comp)
        return components


def demo_graph() -> None:
    # Undirected unweighted
    g = Graph(directed=False)
    edges = [("A","B"),("A","C"),("B","D"),("C","D"),("D","E")]
    for u, v in edges: g.add_edge(u, v)
    assert g.bfs("A") == ["A","B","C","D","E"]
    path = g.shortest_path_bfs("A","E")
    assert path[0] == "A" and path[-1] == "E" and len(path) == 4

    # Directed + topological sort
    dag = Graph(directed=True)
    for u, v in [("A","B"),("A","C"),("B","D"),("C","D"),("D","E")]:
        dag.add_edge(u, v)
    topo = dag.topological_sort()
    assert topo.index("A") < topo.index("B") < topo.index("D")
    assert not dag.has_cycle_directed()
    # Add a cycle
    dag.add_edge("E","A")
    assert dag.has_cycle_directed()

    # Dijkstra
    wg = Graph(directed=True)
    for u,v,w in [("A","B",4),("A","C",2),("C","B",1),("B","D",5),("C","D",8)]:
        wg.add_edge(u, v, w)
    dist, prev = wg.dijkstra("A")
    # A→C=2, C→B=1 → A→C→B=3 (shorter than direct A→B=4)
    # A→C→B→D = 3+5=8; A→C→D = 2+8=10 → shortest D=8
    assert dist["D"] == 8
    path_d = wg.path_to(prev, "A", "D")
    assert path_d == ["A","C","B","D"]

    # Connected components
    ug = Graph(directed=False)
    ug.add_edge(1,2); ug.add_edge(2,3)
    ug.add_edge(4,5)
    ug.add_node(6)
    comps = ug.connected_components()
    assert len(comps) == 3     # [1,2,3], [4,5], [6]

    print("Part 11 (Graph): all assertions passed ✓")

## MAIN

In [ ]:
def do_something(): pass  # placeholder used in demo_primitives

def main() -> None:
    print("=" * 70)
    print("DATA STRUCTURES COMPLETE — python data_structures_complete.py")
    print("=" * 70)
    demo_primitives()
    demo_list()
    demo_tuple()
    demo_set()
    demo_dict()
    demo_collections()
    demo_stack_queue()
    demo_linked_lists()
    demo_bst()
    demo_avl()
    demo_heap()
    demo_trie()
    demo_hashmap()
    demo_graph()
    print("-" * 70)
    print("All data structure demos passed ✔")


if __name__ == "__main__":
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8")
    main()